# Combined plotting notebook

Indlæser data fra `pinn_all_methods_exported_data` og laver alle plots: L2-konvergens vs epochs/tid, residual-grid og heatmap-grid.

In [ ]:

# KOMMENTAR: Denne celle importerer plottingbiblioteker og definerer datamappe, figurmappe og metodenavne.
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# KOMMENTAR: DATA_DIR bestemmer, hvor alle eksporterede .txt/.npy-filer læses fra eller gemmes.
DATA_DIR = Path('pinn_all_methods_exported_data')
FIG_DIR = DATA_DIR / 'figures_all_methods'
FIG_DIR.mkdir(exist_ok=True)
SAVE_FIGURES = True

METHODS = [
    ('baseline', 'Baseline PINN'),
    ('mesh', 'Adaptive refinement'),
    ('finite_difference', 'Finite difference'),
    ('fourier_features', 'Fourier features'),
    ('softadapt', 'SoftAdapt'),
    ('MoE', 'MoE'),
]
v = 1.0
c = 1.0


In [ ]:

# KOMMENTAR: Denne celle definerer robuste indlæsnings- og sorteringsfunktioner til de eksporterede datafiler.
# KOMMENTAR: Returnerer mulige filnavne for samme datasæt, så både .txt, .csv og .npy kan læses.
def candidate_paths(name):
    return [DATA_DIR/name, DATA_DIR/f'{name}.txt', DATA_DIR/f'{name}.csv', DATA_DIR/f'{name}.npy']

# KOMMENTAR: Finder og indlæser én eksportfil fra DATA_DIR.
def load_array(name, required=True):
    for path in candidate_paths(name):
        if path.exists():
            return np.load(path) if path.suffix == '.npy' else np.loadtxt(path)
    if required:
        raise FileNotFoundError(f'Kunne ikke finde {name}')
    return None

# KOMMENTAR: Hvis data er seedvise, tages gennemsnittet over seeds før plotting.
def nanmean_over_seeds(arr):
    arr = np.asarray(arr)
    return arr if arr.ndim == 1 else np.nanmean(arr, axis=0)

# KOMMENTAR: Sorterer punkter efter x-aksen, så kurverne ikke forbindes i forkert rækkefølge.
def sort_by_x(x, y):
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    order = np.argsort(x)
    return x[order], y[order]

# KOMMENTAR: Finder det første tilgængelige evalueringsgrid blandt metodefilerne.
def load_grid():
    for prefix, _ in METHODS:
        x_grid = load_array(f'{prefix}_x_grid', required=False)
        t_grid = load_array(f'{prefix}_t_grid', required=False)
        if x_grid is not None and t_grid is not None:
            return x_grid, t_grid, prefix
    raise FileNotFoundError('Kunne ikke finde x_grid/t_grid.')


## Relativ L2-fejl vs. epochs

In [ ]:

# KOMMENTAR: Denne celle plotter den gennemsnitlige relative L2-fejl som funktion af epochs for alle metoder.
# KOMMENTAR: Opretter figur til én samlet konvergensgraf.
plt.figure(figsize=(9, 5.5))
loaded = []
# KOMMENTAR: Gennemløber alle metoder og plotter kun dem, hvor de nødvendige filer findes.
for prefix, label in METHODS:
    epochs = load_array(f'{prefix}_epochs_eval', required=False)
    l2 = load_array(f'{prefix}_l2_error_vs_epochs', required=False)
    if epochs is None or l2 is None:
        print(f'Springer {label} over: mangler data.')
        continue
    x, y = sort_by_x(epochs, nanmean_over_seeds(l2))
    plt.semilogy(x, y, linewidth=2, label=label)
    loaded.append(label)
plt.xlabel('Epochs')
plt.ylabel(r'Relativ $L^2$-fejl')
plt.title('Konvergens vs. epochs')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.tight_layout()
if SAVE_FIGURES: plt.savefig(FIG_DIR/'relative_L2_vs_epochs_all_methods.png', dpi=300, bbox_inches='tight')
plt.show()
print('Metoder plottet:', loaded)


## Relativ L2-fejl vs. wall-clock tid

In [ ]:

# KOMMENTAR: Denne celle plotter den gennemsnitlige relative L2-fejl som funktion af wall-clock tid for alle metoder.
# KOMMENTAR: Opretter figur til én samlet konvergensgraf.
plt.figure(figsize=(9, 5.5))
loaded = []
# KOMMENTAR: Gennemløber alle metoder og plotter kun dem, hvor de nødvendige filer findes.
for prefix, label in METHODS:
    times = load_array(f'{prefix}_actual_times', required=False)
    l2_time = load_array(f'{prefix}_l2_error_time', required=False)
    if times is None or l2_time is None:
        print(f'Springer {label} over: mangler data.')
        continue
    x, y = sort_by_x(nanmean_over_seeds(times), nanmean_over_seeds(l2_time))
    plt.semilogy(x, y, linewidth=2, label=label)
    loaded.append(label)
plt.xlabel('Tid [s]')
plt.ylabel(r'Relativ $L^2$-fejl')
plt.title('Konvergens vs. wall-clock tid')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.tight_layout()
if SAVE_FIGURES: plt.savefig(FIG_DIR/'relative_L2_vs_time_all_methods.png', dpi=300, bbox_inches='tight')
plt.show()
print('Metoder plottet:', loaded)


## Residual-/loss-grid

In [ ]:
# KOMMENTAR: Denne celle plotter PDE-, initial- og boundary-loss i et 2x3 grid.
# KOMMENTAR: Opretter et 2x3 grid, så de seks metoder vises i samme figur.
fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)
axes = axes.ravel()

missing = []
for ax, (prefix, label) in zip(axes, METHODS):
    epochs = load_array(f'{prefix}_epochs_eval', required=False)
    pde = load_array(f'{prefix}_loss_pde_vs_epochs', required=False)
    ini = load_array(f'{prefix}_loss_ini_vs_epochs', required=False)
    bc = load_array(f'{prefix}_loss_bc_vs_epochs', required=False)

    if epochs is None or pde is None or ini is None or bc is None:
        ax.text(0.5, 0.5, 'Mangler\nloss-filer',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(label)
        ax.axis('off')
        missing.append(label)
        continue

    for arr, component in [(pde, 'PDE'), (ini, 'Initial'), (bc, 'Boundary')]:
        x, y = sort_by_x(epochs, nanmean_over_seeds(arr))
        ax.semilogy(x, y, linewidth=1.8, label=component)

    ax.set_title(label)
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(fontsize=8)

fig.suptitle('PDE-, initial- og boundary-loss for hver metode', fontsize=16)
if SAVE_FIGURES:
    fig.savefig(FIG_DIR/'residual_components_grid_all_methods.png',
                dpi=300, bbox_inches='tight')
plt.show()

if missing:
    print('Manglede residual/loss-filer for:', ', '.join(missing))

## Heatmap-grid med eksakt løsning og fejlmaps

In [ ]:
# KOMMENTAR: Denne celle plotter den eksakte løsning og gennemsnitlige punktvise absolutte fejlmaps.
# KOMMENTAR: Indlæser grid'et, som bruges til både eksakt løsning og heatmaps.
x_grid, t_grid, grid_prefix = load_grid()
print('Bruger grid fra:', grid_prefix)

# KOMMENTAR: Beregner den analytiske løsning på evalueringsgrid'et.
U_exact = np.exp(-v*t_grid) * np.sin(x_grid - c*t_grid)
extent = [x_grid.min(), x_grid.max(), t_grid.min(), t_grid.max()]

# Load error maps in the same order as METHODS.
# KOMMENTAR: Samler fejlheatmaps fra de metoder, der har eksporteret dem.
error_maps = []
# KOMMENTAR: Gennemløber alle metoder og plotter kun dem, hvor de nødvendige filer findes.
for prefix, label in METHODS:
    err = load_array(f'{prefix}_enviroment_error', required=False)
    if err is not None:
        error_maps.append((prefix, label, err))
    else:
        print(f'Mangler error heatmap for {label}.')

# Layout:
# [ exact ] [ baseline ] [ mesh      ] [ finite_difference ]
# [ empty ] [ fourier  ] [ softadapt ] [ MoE               ]
fig = plt.figure(figsize=(18, 8), constrained_layout=True)
gs = fig.add_gridspec(2, 4)

ax_exact = fig.add_subplot(gs[0, 0])
ax_empty = fig.add_subplot(gs[1, 0])
ax_empty.axis('off')

# KOMMENTAR: Definerer aksernes placering, så Fourier features ligger under finite difference.
method_axes = [
    fig.add_subplot(gs[0, 1]),  # baseline
    fig.add_subplot(gs[0, 2]),  # mesh/adaptive refinement
    fig.add_subplot(gs[0, 3]),  # finite difference
    fig.add_subplot(gs[1, 1]),  # fourier features
    fig.add_subplot(gs[1, 2]),  # softadapt
    fig.add_subplot(gs[1, 3]),  # MoE
]

im0 = ax_exact.imshow(U_exact, extent=extent, origin='lower', aspect='auto')
ax_exact.set_title(r'Eksakt løsning $u_{exact}$')
ax_exact.set_xlabel('x')
ax_exact.set_ylabel('t')
fig.colorbar(im0, ax=ax_exact, fraction=0.046, pad=0.04, label=r'$u$')

if error_maps:
    vmax_err = np.nanmax([np.nanmax(err) for _, _, err in error_maps])
    ims = []

    for ax, (_, label, err) in zip(method_axes, error_maps):
        im = ax.imshow(
            err,
            extent=extent,
            origin='lower',
            aspect='auto',
            vmin=0.0,
            vmax=vmax_err
        )
        ims.append(im)
        ax.set_title(label)
        ax.set_xlabel('x')
        ax.set_ylabel('t')

    # Hide unused method axes if some methods are missing.
    for ax in method_axes[len(error_maps):]:
        ax.axis('off')

    fig.colorbar(
        ims[-1],
        ax=method_axes[:len(error_maps)],
        fraction=0.025,
        pad=0.02,
        label='Punktvis absolut fejl'
    )
else:
    for ax in method_axes:
        ax.axis('off')

fig.suptitle('Eksakt løsning og gennemsnitlig punktvis absolut fejl', fontsize=16)
if SAVE_FIGURES:
    fig.savefig(FIG_DIR/'heatmaps_exact_and_errors_all_methods.png',
                dpi=300, bbox_inches='tight')
plt.show()

## Ekstra heatmap-grid med PINN-løsninger

In [ ]:
# KOMMENTAR: Denne celle plotter middel-PINN-løsningerne for hver metode i samme layout som fejlmaps.
# KOMMENTAR: Samler de gennemsnitlige PINN-løsninger fra metodefilerne.
solution_maps = []
# KOMMENTAR: Gennemløber alle metoder og plotter kun dem, hvor de nødvendige filer findes.
for prefix, label in METHODS:
    sol = load_array(f'{prefix}_pinn_solution_mean', required=False)
    if sol is not None:
        solution_maps.append((prefix, label, sol))
    else:
        print(f'Mangler solution heatmap for {label}.')

# Same layout as the error heatmaps:
# [ exact ] [ baseline ] [ mesh      ] [ finite_difference ]
# [ empty ] [ fourier  ] [ softadapt ] [ MoE               ]
fig = plt.figure(figsize=(18, 8), constrained_layout=True)
gs = fig.add_gridspec(2, 4)

ax_exact = fig.add_subplot(gs[0, 0])
ax_empty = fig.add_subplot(gs[1, 0])
ax_empty.axis('off')

# KOMMENTAR: Definerer aksernes placering, så Fourier features ligger under finite difference.
method_axes = [
    fig.add_subplot(gs[0, 1]),  # baseline
    fig.add_subplot(gs[0, 2]),  # mesh/adaptive refinement
    fig.add_subplot(gs[0, 3]),  # finite difference
    fig.add_subplot(gs[1, 1]),  # fourier features
    fig.add_subplot(gs[1, 2]),  # softadapt
    fig.add_subplot(gs[1, 3]),  # MoE
]

all_values = [U_exact] + [sol for _, _, sol in solution_maps]
vmin = np.nanmin([np.nanmin(a) for a in all_values])
vmax = np.nanmax([np.nanmax(a) for a in all_values])

im0 = ax_exact.imshow(
    U_exact,
    extent=extent,
    origin='lower',
    aspect='auto',
    vmin=vmin,
    vmax=vmax
)
ax_exact.set_title(r'Eksakt løsning $u_{exact}$')
ax_exact.set_xlabel('x')
ax_exact.set_ylabel('t')

ims = [im0]
for ax, (_, label, sol) in zip(method_axes, solution_maps):
    im = ax.imshow(
        sol,
        extent=extent,
        origin='lower',
        aspect='auto',
        vmin=vmin,
        vmax=vmax
    )
    ims.append(im)
    ax.set_title(label)
    ax.set_xlabel('x')
    ax.set_ylabel('t')

# Hide unused method axes if some methods are missing.
for ax in method_axes[len(solution_maps):]:
    ax.axis('off')

fig.colorbar(
    ims[-1],
    ax=[ax_exact] + method_axes[:len(solution_maps)],
    fraction=0.025,
    pad=0.02,
    label=r'$u$'
)

fig.suptitle('Eksakt løsning og gennemsnitlige PINN-løsninger', fontsize=16)
if SAVE_FIGURES:
    fig.savefig(FIG_DIR/'heatmaps_exact_and_solutions_all_methods.png',
                dpi=300, bbox_inches='tight')
plt.show()